In [4]:
MODULE_DIR = '/Users/ravitpichayavet/Documents/GaTechIE/GraduateResearch/CTC_CVRP/Modules'
GUROBI_LICENSE_DIR = '/Users/ravitpichayavet/gurobi.lic'

ARG_COLOR = '#104375'
DEPOT_COLOR = '#D0F6FF'
NODE_COLOR = '#484848'

import sys
sys.path.insert(0,MODULE_DIR)
import importlib
from datetime import datetime

import visualize_sol as vis_sol
import initialize_path as init_path
import random_instance as rand_inst
import utility as util
import bnp as bnp
import model as md

import numpy as np
from gurobipy import *
import os
os.environ['GRB_LICENSE_FILE'] = GUROBI_LICENSE_DIR


from itertools import combinations,permutations 
import random 
import pandas as pd
import itertools
import plotly.graph_objects as go
import networkx as nx
import plotly.offline as py 
import pickle as pk
import nltk
import time
import copy

from matplotlib import pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from scipy.spatial import distance
import logging

import pybnb
import numpy as np
import gurobipy as gp
from copy import deepcopy
from math import ceil
# logging.basicConfig(filename='myapp.log',filemode='w',format='%(message)s',level=logging.INFO)
# print = logging.info

In [5]:
NO_DEMAND_NODE = 8
DOM_RULE = 4

truck_capacity = 20
fixed_setup_time = 0 # fixed_setup_time = 0.5
truck_speed = 20
max_vehicles = None
#Initial set R+
max_nodes_proute = 3
max_vehicles_proute = np.inf
time_window = 2 #hour
tw_avg_factor = 1 #0.5 = avg, 1 = worst case waiting

max_nodes_proute_DP = NO_DEMAND_NODE
max_vehicles_proute_DP = max_vehicles_proute
m_col_list = None #[i for i in range(1,max_vehicles_proute_DP+1)]
# DP_MODE = "SIMUL_M"
DP_TIME_LIM = np.inf
#################################################################
constant_dict = { 'truck_capacity' : truck_capacity,
                 'fixed_setup_time' : fixed_setup_time,
                'truck_speed': truck_speed,
                 'max_vehicles':max_vehicles,
                'max_nodes_proute':max_nodes_proute,
                'max_vehicles_proute':max_vehicles_proute,
                'max_nodes_proute_DP':max_nodes_proute_DP,
                'max_vehicles_proute_DP':max_vehicles_proute_DP,
                'time_window':time_window,
                'tw_avg_factor':tw_avg_factor}

edge_plot_config = {'line_width':1.5, 'line_color':ARG_COLOR, 'dash':None, 'name':''}
############# Create network components: nodes, arcs ############
labeling_dict = vis_sol.create_nodes(0,NO_DEMAND_NODE)
docking,customers,depot,depot_s,depot_t,all_depot,nodes,arcs = labeling_dict.values()

### (1) Import Instance

In [6]:
INST_TYPE = 1
file_no = 10
suffix = 'L2norm'
TYPE_DIR = '../ComputationalExperiment/InstancesForExperiment/L2_norm/TYPE{0}/{1}N/'.format(INST_TYPE,NO_DEMAND_NODE)

INST_NAME = 'InstanceType{0}_{1}n_{2}_{3}'.format(INST_TYPE,
                                                NO_DEMAND_NODE,
                                                file_no,suffix);
print(INST_NAME)

InstanceType1_8n_10_L2norm


In [7]:
_distMat,_nodeTce,_cusDem,_nodePos = \
            rand_inst.import_instance(TYPE_DIR,INST_NAME)

Importing instance: InstanceType1_8n_10_L2norm


In [10]:
vis_sol.plot_network([], _nodeTce,_cus_dem=_cusDem,
                     _title=INST_NAME,_display_cus_dem=True)

### Column Generation with Initial Set (npr = 3)

In [14]:
constant_dict['max_nodes_proute'] = 3
constant_dict['init_max_nodes_proute'] = 3

In [15]:
_nodeTce['marker']['size'] = 10
labeling_dict = vis_sol.create_nodes(0,len(_nodePos)-1)
docking,customers,depot,depot_s,depot_t,all_depot,nodes,arcs = labeling_dict.values()
#INIT R+
_initializer = init_path.InitialRouteGenerator(1,labeling_dict,
                                      _cusDem,constant_dict,
                                      _distMat)
_row_labels = ['lr','m']+depot+customers+arcs
_init_route = _initializer.generateInitDFV4wTimeWindow(_row_labels,constant_dict)

Generate Init cols with time window: 2
progress: 1
progress: 10
progress: 100
Elapsed Time: 0.35037922859191895


/Users/ravitpichayavet/Documents/GaTechIE/GraduateResearch/CTC_CVRP/Modules/initialize_path.py:337: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.1927863576486026' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.

/Users/ravitpichayavet/Documents/GaTechIE/GraduateResearch/CTC_CVRP/Modules/initialize_path.py:345: RuntimeWarning:

divide by zero encountered in log10

/Users/ravitpichayavet/Documents/GaTechIE/GraduateResearch/CTC_CVRP/Modules/initialize_path.py:337: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1.4873524222107466' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.

/Users/ravitpichayavet/Documents/GaTechIE/GraduateResearch/CTC_CVRP/Modules/initialize_path.py:337: FutureWarning:

Setting an item of incompatible dtype is deprecated an

In [16]:
tWLPrelax_model = md.timeWindowModel(_init_route, _initializer,
             _distMat,constant_dict, _relax_route=True)
tWLPrelax_model.buildModel()
tWLPrelax_model.model.setParam('OutputFlag',True)
tWLPrelax_model.solveRelaxedModel()
tWLPrelax_model.relaxedModel.ObjVal

[(v.varName, v.x) for v in tWLPrelax_model.relaxedModel.getVars() if v.x>0]
# tWLPrelax_model.init_routes_df
# tWLPrelax_model.shortCuttingColumns()
print("IP model")
tWLPrelax_model.solveModel()
print(tWLPrelax_model.model.ObjVal,[(v.varName, round(v.x)) for v in tWLPrelax_model.model.getVars() if v.x>0])

# tWLPrelax_model.solveRelaxedBoundedModel()
# tWLPrelax_model.relaxedBoundedModel.ObjVal

tW_sol = tWLPrelax_model.getRouteSolution(tWLPrelax_model.model.getVars(),
                                             edge_plot_config,_nodeTce,
                                              tWLPrelax_model.customer_demand)
vis_sol.plot_network(tW_sol,_nodeTce,_display_cus_dem=True,
                            _cus_dem=_cusDem,_title="TBA",
                        _display_plot=True,_display_info_table=True,
                     _show_all_info=False)


Set parameter Username
Academic license - for non-commercial use only - expires 2026-03-19
Finish generating variables!
Finish generating constrains!
Finish generating objective!
Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 24.6.0 24G84)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 8 rows, 388 columns and 1092 nonzeros
Model fingerprint: 0x976e3f63
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve removed 0 rows and 303 columns
Presolve time: 0.00s
Presolved: 8 rows, 85 columns, 213 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   8.000000e+00   0.000000e+00      0s
      13    4.7500000e+00   0.000000e+00   0.000000e+00      0s

Solved in 13 iterations and 0.01 seconds (0.00 work un

/Users/ravitpichayavet/Documents/GaTechIE/GraduateResearch/CTC_CVRP/Modules/model.py:1412: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

/Users/ravitpichayavet/Documents/GaTechIE/GraduateResearch/CTC_CVRP/Modules/model.py:1413: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

/Users/ravitpichayavet/Documents/GaTechIE/GraduateResearch/CTC_CVRP/Modules/model.py:1805: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

/Users/ravitpichayavet/Do

NameError: name 'operator' is not defined

In [9]:
tWLPrelax_model.runColumnsGeneration(None,
                    _pricing_status=True,_check_dominance=True,
                    _dominance_rule=4,_DP_ver="SIMUL_M",
                    _time_limit=np.inf,_update_m_ub=False,
                    _filtering_mode="BestRwdPerI")
# tWLPrelax_model.relaxedModel.ObjVal
tWLPrelax_model.relaxedModel.ObjVal

.Running Col. Gen. with DP mode:  SIMUL_M
Gurobi Optimizer version 9.1.1 build v9.1.1rc0 (mac64)
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 8 rows, 388 columns and 1092 nonzeros
Model fingerprint: 0x976e3f63
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve time: 0.01s
Presolved: 8 rows, 388 columns, 1092 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   8.000000e+00   0.000000e+00      0s
      13    4.7500000e+00   0.000000e+00   0.000000e+00      0s

Solved in 13 iterations and 0.01 seconds
Optimal objective  4.750000000e+00
.Start Running DP | Max nodes visited: 8 | Max vehicles per route: inf | Out-loop-0

 DUALS: [0.25, 1.0, 0.5, 0.5, 1.0, 0.75, 0.0, 0.75] mMAX: inf
Solving time limit set to: inf secs. Dominance Checking: True
Dominance Version: 4
l-threshol

Thread count: 8 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 8 rows, 401 columns and 1135 nonzeros
Model fingerprint: 0x82f36051
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [1e+00, 1e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve time: 0.00s
Presolved: 8 rows, 401 columns, 1135 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   3.750000e+00   0.000000e+00      0s
      14    3.6111111e+00   0.000000e+00   0.000000e+00      0s

Solved in 14 iterations and 0.00 seconds
Optimal objective  3.611111111e+00
.Start Running DP | Max nodes visited: 8 | Max vehicles per route: inf | Out-loop-2

 DUALS: [0.2222222222222222, 0.8888888888888888, 0.5, 0.0, 1.1111111111111112, 0.3333333333333333, 0.0, 0.5555555555555556] mMAX: inf
Solving time limit set to: inf secs. Dominance Checking: True
Dominance Version: 4
l-threshold:  1.5
States explored i

3.5

In [10]:
# tWLPrelax_model.init_routes_df
# tWLPrelax_model.shortCuttingColumns()
tWLPrelax_model.solveModel()
print(tWLPrelax_model.model.ObjVal,[(v.varName, round(v.x)) for v in tWLPrelax_model.model.getVars() if v.x>0])

tW_sol = tWLPrelax_model.getRouteSolution(tWLPrelax_model.model.getVars(),
                                             edge_plot_config,_nodeTce,
                                              tWLPrelax_model.customer_demand)
vis_sol.plot_network(tW_sol,_nodeTce,_display_cus_dem=True,
                            _cus_dem=_cusDem,_title="TBA",
                        _display_plot=True,_display_info_table=True,
                     _show_all_info=False)

Parameter PoolSearchMode unchanged
   Value: 2  Min: 0  Max: 2  Default: 0
Gurobi Optimizer version 9.1.1 build v9.1.1rc0 (mac64)
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 8 rows, 404 columns and 1143 nonzeros
Model fingerprint: 0x34c099cb
Variable types: 0 continuous, 404 integer (404 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [1e+00, 1e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

MIP start from previous solve produced solution with objective 5 (0.01s)
Loaded MIP start from previous solve with objective 5

Presolve time: 0.00s
Presolved: 8 rows, 404 columns, 1143 nonzeros
Variable types: 0 continuous, 404 integer (404 binary)

Root relaxation: objective 4.750000e+00, 14 iterations, 0.00 seconds

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0    

### Full Enumeration with Initial Set (npr = no. customer = 8)

In [11]:
constant_dict['max_nodes_proute'] = NO_DEMAND_NODE

In [12]:
_nodeTce['marker']['size'] = 10
labeling_dict = vis_sol.create_nodes(0,len(_nodePos)-1)
docking,customers,depot,depot_s,depot_t,all_depot,nodes,arcs = labeling_dict.values()
#INIT R+
_initializer = init_path.InitialRouteGenerator(1,labeling_dict,
                                      _cusDem,constant_dict,
                                      _distMat)
_row_labels = ['lr','m']+depot+customers+arcs
_init_route = _initializer.generateInitDFV4wTimeWindow(_row_labels,constant_dict)

Generate Init cols with time window: 2
progress: 1
progress: 10
progress: 100


/Users/ravitpichayavet/Documents/GaTechIE/GraduateResearch/CTC_CVRP/Modules/initialize_path.py:342: RuntimeWarning:

divide by zero encountered in log10



progress: 1000
Elapsed Time: 24.35154700279236


In [13]:
tWFullEnu_model = md.timeWindowModel(_init_route, _initializer,
             _distMat,constant_dict, _relax_route=False)
tWFullEnu_model.buildModel()
tWFullEnu_model.model.setParam('OutputFlag',True)
tWFullEnu_model.solveRelaxedModel()
tWFullEnu_model.relaxedModel.ObjVal

[(v.varName, v.x) for v in tWFullEnu_model.relaxedModel.getVars() if v.x>0]
# tWLPrelax_model.init_routes_df
# tWLPrelax_model.shortCuttingColumns()
print("IP model Full Enumeration")
tWFullEnu_model.solveModel()
tWFullEnu_model.model.ObjVal,[(v.varName, round(v.x)) for v in tWFullEnu_model.model.getVars() if v.x>0]

# tWLPrelax_model.solveRelaxedBoundedModel()
# tWLPrelax_model.relaxedBoundedModel.ObjVal

tWFullEnu_sol = tWFullEnu_model.getRouteSolution(tWFullEnu_model.model.getVars(),
                                             edge_plot_config,_nodeTce,
                                              tWFullEnu_model.customer_demand)
vis_sol.plot_network(tWFullEnu_sol,_nodeTce,_display_cus_dem=True,
                            _cus_dem=_cusDem,_title="FullEnumeration",
                        _display_plot=True,_display_info_table=True,
                     _show_all_info=False)


Finish generating variables!
Finish generating constrains!
Finish generating objective!
Parameter OutputFlag unchanged
   Value: 1  Min: 0  Max: 1  Default: 1
Gurobi Optimizer version 9.1.1 build v9.1.1rc0 (mac64)
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 8 rows, 5004 columns and 24510 nonzeros
Model fingerprint: 0x161d9c94
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 5e+04]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve removed 0 rows and 4808 columns
Presolve time: 0.01s
Presolved: 8 rows, 196 columns, 724 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.0000000e+00   6.000000e+00   0.000000e+00      0s
      10    4.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 10 iterations and 0.02 seconds
Optimal objective  4.000000000e+00
IP model Full Enumeration
Changed value of parameter PoolSearchMode to 2
   Prev:

## BnP


In [5]:
NO_DEMAND_NODE = 8
DOM_RULE = 4

truck_capacity = 20
fixed_setup_time = 0 # fixed_setup_time = 0.5
truck_speed = 20
max_vehicles = None
#Initial set R+
max_nodes_proute = 3
max_vehicles_proute = np.inf
time_window = 2 #hour
tw_avg_factor = 1 #0.5 = avg, 1 = worst case waiting

max_nodes_proute_DP = NO_DEMAND_NODE
max_vehicles_proute_DP = max_vehicles_proute
m_col_list = None #[i for i in range(1,max_vehicles_proute_DP+1)]
# DP_MODE = "SIMUL_M"
DP_TIME_LIM = np.inf
#################################################################
constant_dict = { 'truck_capacity' : truck_capacity,
                 'fixed_setup_time' : fixed_setup_time,
                'truck_speed': truck_speed,
                 'max_vehicles':max_vehicles,
                'max_nodes_proute':max_nodes_proute,
                'max_vehicles_proute':max_vehicles_proute,
                'max_nodes_proute_DP':max_nodes_proute_DP,
                'max_vehicles_proute_DP':max_vehicles_proute_DP,
                'time_window':time_window,
                'tw_avg_factor':tw_avg_factor}

edge_plot_config = {'line_width':1.5, 'line_color':ARG_COLOR, 'dash':None, 'name':''}
############# Create network components: nodes, arcs ############
labeling_dict = vis_sol.create_nodes(0,NO_DEMAND_NODE)
docking,customers,depot,depot_s,depot_t,all_depot,nodes,arcs = labeling_dict.values()

### (1) Import Instance

INST_TYPE = 1
file_no = 10
suffix = 'L2norm'
TYPE_DIR = '../ComputationalExperiment/InstancesForExperiment/L2_norm/TYPE{0}/{1}N/'.format(INST_TYPE,NO_DEMAND_NODE)

INST_NAME = 'InstanceType{0}_{1}n_{2}_{3}'.format(INST_TYPE,
                                                NO_DEMAND_NODE,
                                                file_no,suffix);
print(INST_NAME);
_distMat,_nodeTce,_cusDem,_nodePos = \
            rand_inst.import_instance(TYPE_DIR,INST_NAME)

InstanceType1_8n_10_L2norm
Importing instance: InstanceType1_8n_10_L2norm


In [11]:
m1 = sys.modules['visualize_sol']
m2 = sys.modules['random_instance']
m3 = sys.modules['initialize_path']
m4 = sys.modules['model']
m5 = sys.modules['bnp']
importlib.reload(m1)
importlib.reload(m2)
importlib.reload(m3)
importlib.reload(m4)
importlib.reload(m5)

<module 'bnp' from '/Users/ravitpichayavet/Documents/GaTechIE/GraduateResearch/CTC_CVRP/Modules/bnp.py'>

In [12]:
_nodeTce['marker']['size'] = 10
labeling_dict = vis_sol.create_nodes(0,len(_nodePos)-1)
docking,customers,depot,depot_s,depot_t,all_depot,nodes,arcs = labeling_dict.values()
#INIT R+
_initializer = init_path.InitialRouteGenerator(1,labeling_dict,
                                      _cusDem,constant_dict,
                                      _distMat)
_row_labels = ['lr','m']+depot+customers+arcs
_init_route = _initializer.generateInitDFV4wTimeWindow(_row_labels,constant_dict)

Generate Init cols with time window: 2
progress: 1
progress: 10
progress: 100
Elapsed Time: 0.1672530174255371


/Users/ravitpichayavet/Documents/GaTechIE/GraduateResearch/CTC_CVRP/Modules/initialize_path.py:342: RuntimeWarning:

divide by zero encountered in log10



In [13]:
problem = bnp.cTCCVRP_mt(_distMat, _initializer, 
                         _init_route, constant_dict,False, True)
solver = pybnb.Solver(comm=None)
results = solver.solve(problem, absolute_gap=1e-6)
# opt = verify_sol(bp_instance)
print(results.objective,)

Finish generating variables!
Finish generating constrains!
Finish generating objective!

Using non-default solver options:
 - absolute_gap: 1e-06 (default: 0)

Starting branch & bound solve:
 - dispatcher pid: 7168 (Ravits-MBP.attlocal.net)
 - worker processes: 1
------------------------------------------------------------------------------------------------------------------------
         Nodes        |                     Objective Bounds                      |              Work              
      Expl    Unexpl  |      Incumbent           Bound    Rel. Gap       Abs. Gap | Time (s)  Nodes/Sec Imbalance   Idle
         0         1  |            inf            -inf         inf%           inf |      0.0       0.00     0.00%      0
incident_dict {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0}
==Model's status: 2
==THIS IS NODE: 0
.Running Col. Gen. with DP mode:  SIMUL_M

 DUALS: [0.25, 1.0, 0.5, 0.5, 1.0, 0.75, -0.0, 0.75] mMAX: inf
Solving time limit set to: inf secs. Dominan

==Obj-val colgen :bf_sh/af_sh: 3.5 3.5
==Del Pats: ['sDP_CBnP0-31-8', 'route[385]', 'route[384]', 'route[379]', 'route[378]', 'route[373]', 'route[372]', 'route[369]', 'route[368]', 'route[367]', 'route[366]', 'route[365]', 'route[364]', 'route[363]', 'route[362]', 'route[361]', 'route[359]', 'route[358]', 'route[354]', 'route[353]', 'route[348]', 'route[347]', 'route[343]', 'route[342]', 'route[337]', 'route[336]', 'route[331]', 'route[330]', 'route[327]', 'route[326]', 'route[325]', 'route[324]', 'route[323]', 'route[322]', 'route[321]', 'route[320]', 'route[319]', 'route[317]', 'route[316]', 'route[312]', 'route[311]', 'route[306]', 'route[305]', 'route[301]', 'route[300]', 'route[295]', 'route[294]', 'route[289]', 'route[288]', 'route[285]', 'route[284]', 'route[283]', 'route[282]', 'route[281]', 'route[280]', 'route[279]', 'route[278]', 'route[277]', 'route[275]', 'route[274]', 'route[270]', 'route[269]', 'route[264]', 'route[263]', 'route[259]', 'route[258]', 'route[249]', 'route

incident_dict {0: 0, 1: 1, 2: 0, 3: 1, 4: 1, 5: 0, 6: 0, 7: 0, 8: 1}
==Model's status: 2
==THIS IS NODE: 4
.Running Col. Gen. with DP mode:  SIMUL_M

 DUALS: [0.3333333333333333, 1.6666666666666665, 0.0, 0.5, 0.3333333333333335, 0.33333333333333337, -0.0, 0.33333333333333337] mMAX: inf
Solving time limit set to: inf secs. Dominance Checking: True
Dominance Version: 4
bch-conds: [[('c_3', 'c_4'), 1], [('c_1', 'c_8'), 1]]
forbidden_link: {0: [4, 8], 1: [4, 0, 2, 3, 5, 6, 7], 2: [4, 8], 3: [0, 1, 2, 5, 6, 7, 8], 4: [8], 5: [4, 8], 6: [4, 8], 7: [4, 8], 8: [4]}
necessary_link: {0: [], 1: [8], 2: [], 3: [4], 4: [], 5: [], 6: [], 7: [], 8: []}
States explored in 0-iters:(623, 1033)

Reconstructing Path....
 Filtering Mode: BestRwdPerI
[([0, 0], 0), ([0, 2, 6, 1, 8, 1, 0], 4.440892098500626e-16), ([0, 6, 2, 0], 0.0), ([0, 3, 4, 3, 4, 3, 4, 3, 0], -0.5), ([0, 3, 4, 3, 4, 3, 4, 3, 4, 0], 0.0), ([0, 2, 5, 0], 0.0), ([0, 6, 2, 6, 0], 0.3333333333333335), ([0, 2, 7, 0], -0.3333333333333335), ([0, 

==Obj-val colgen :bf_sh/af_sh: 4.166666666666666 4.166666666666666
==Del Pats: ['sDP_CBnP3-80-6', 'sDP_CBnP3-70-6', 'sDP_CBnP3-60-1', 'sDP_CBnP3-20-6', 'sDP_CBnP3-10-8', 'route[351]', 'route[350]', 'route[349]', 'route[346]', 'route[340]', 'route[298]', 'route[256]', 'route[139]', 'route[57]']
[['route[7]', 1.0, {('O', 'c_8'): 1.0, ('c_8', 'O'): 1.0}], ['route[313]', 1.0, {('c_5', 'O'): 1.0, ('O', 'c_7'): 1.0, ('c_2', 'c_5'): 1.0, ('c_7', 'c_2'): 1.0}], ['sDP_CBnP0-10-6', 0.3333333333333333, {('O', 'c_6'): 1.0, ('c_1', 'O'): 1.0, ('c_6', 'c_1'): 3.0, ('c_1', 'c_6'): 2.0}], ['sDP_CBnP0-40-3', 0.25, {('O', 'c_3'): 1.0, ('c_4', 'O'): 1.0, ('c_3', 'c_4'): 4.0, ('c_4', 'c_3'): 3.0}]]
==Called bound(), Update LOCAL BOUND: 4.166666666666666
shortcutting columns: 0 / 21
LP/IP: 4.166666666666666 5.0
==Called branch, branching with cond: [[('c_3', 'c_4'), 1], [('c_1', 'c_8'), 0], [('c_8', 'c_1'), 0]]
===Len lp_sol: 409 Len route_pats: 409
r_w_cycle ['sDP_CBnP0-40-3', 'sDP_CBnP0-10-6']
bch_arcs_d

RouteName: sDP_CBnP9-80-6 Route: [0, 8, 1, 6, 8, 0] RouteCost: 2 Reward: 0.6666666666666665

 DUALS: [0.16666666666666666, 0.6666666666666667, 1.0, -0.0, 1.3333333333333333, 0.3333333333333333, -0.0, 0.6666666666666667] mMAX: inf
Solving time limit set to: inf secs. Dominance Checking: True
Dominance Version: 4
bch-conds: [[('c_3', 'c_4'), 0], [('c_1', 'c_8'), 0]]
forbidden_link: {0: [], 1: [8], 2: [], 3: [4], 4: [], 5: [], 6: [], 7: [], 8: []}
necessary_link: {0: [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: [], 7: [], 8: []}
States explored in 1-iters:(1657, 3090)

Reconstructing Path....
 Filtering Mode: BestRwdPerI
[([0, 0], 0), ([0, 8, 1, 6, 8, 1, 0], 0.0), ([0, 8, 5, 2, 0], -0.33333333333333304), ([0, 3, 0], 0.0), ([0, 4, 0], -1.0), ([0, 2, 5, 0], 0.0), ([0, 3, 8, 1, 6, 0], 0.16666666666666696), ([0, 6, 8, 1, 6, 7, 0], -0.5000000000000002), ([0, 8, 1, 6, 8, 0], -0.16666666666666652)]
RWDList: [False, False, False, False, False, False, True, False, False]
RouteName: sDP_CBnP9-61-1 Rou

LP/IP: 4.0 5.0
==Called branch, branching with cond: [[('c_3', 'c_4'), 0], [('c_1', 'c_8'), 1]]
===Len lp_sol: 404 Len route_pats: 404
r_w_cycle ['sDP_CBnP0-62-8']
bch_arcs_dict {('O', 'c_6'): -9999999999.5, ('c_6', 'O'): -9999999999.5, ('c_8', 'c_6'): 0.5, ('c_6', 'c_1'): 0.5, ('c_8', 'c_1'): 0.5}
frac_dict {('O', 'c_6'): -9999999999.5, ('c_6', 'O'): -9999999999.5, ('c_8', 'c_6'): 0.5, ('c_6', 'c_1'): 0.5, ('c_8', 'c_1'): 0.5}
frac ('c_8', 'c_6')
===branching on arc/val ('c_8', 'c_6')
incident_dict {0: 0, 1: 1, 2: 0, 3: 1, 4: 1, 5: 0, 6: 0, 7: 0, 8: 1}
==Model's status: 2
==THIS IS NODE: 11
.Running Col. Gen. with DP mode:  SIMUL_M

 DUALS: [0.08333333333333326, 0.8333333333333335, 0.0, 1.0, 1.0833333333333333, 0.5833333333333333, 0.08333333333333334, 0.5833333333333335] mMAX: inf
Solving time limit set to: inf secs. Dominance Checking: True
Dominance Version: 4
bch-conds: [[('c_3', 'c_4'), 1], [('c_1', 'c_8'), 1], [('c_4', 'c_3'), 0], [('c_8', 'c_6'), 0]]
forbidden_link: {0: [4, 8], 